# EN4554 Assignment 2 — Convolutional Neural Networks

**Name:** _your name_
**Index number:** _your index_
**GitHub repository:** _link_

**AI tool acknowledgement:** Claude (Anthropic) was used as a coding and learning assistant for this assignment. All code was reviewed, run, and understood by me.

## Setup

* Dataset: CIFAR-10, 50k training images split into **45k train / 5k validation** (fixed random permutation).
* Normalisation: per-channel mean and std computed on the **45k training images only**, then applied to train and validation. Computing statistics on validation or test data would leak information about the data we evaluate on.
* Seeds: Python, NumPy and PyTorch seeds are fixed so the results are reproducible.
* The test set is **not loaded here**. It is used once, in Task 4.

The whole dataset is small (45k × 3 × 32 × 32 floats ≈ 550 MB), so it is kept on the GPU as one tensor and mini-batches are taken by indexing. This is much faster than a `DataLoader` for CIFAR-sized data.

In [ ]:
import math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 0
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device,
      '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

In [ ]:
train_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
classes = train_full.classes

# uint8 (50000, 32, 32, 3)  ->  float (50000, 3, 32, 32) in [0, 1]
X_all = torch.tensor(train_full.data).permute(0, 3, 1, 2).float() / 255.0
Y_all = torch.tensor(train_full.targets)

# Fixed 45k / 5k split
perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]

# Per-channel statistics from the TRAINING split only
mean = X_all[train_idx].mean(dim=(0, 2, 3), keepdim=True)
std  = X_all[train_idx].std(dim=(0, 2, 3), keepdim=True)

X_train = ((X_all[train_idx] - mean) / std).to(device)
Y_train = Y_all[train_idx].to(device)
X_val   = ((X_all[val_idx] - mean) / std).to(device)
Y_val   = Y_all[val_idx].to(device)

print('train:', tuple(X_train.shape), ' val:', tuple(X_val.shape))
print('channel mean:', mean.flatten().numpy().round(4))
print('channel std :', std.flatten().numpy().round(4))
print('normalised train mean/std per channel:',
      X_train.mean(dim=(0, 2, 3)).cpu().numpy().round(3), X_train.std(dim=(0, 2, 3)).cpu().numpy().round(3))

In [ ]:
def batches(X, Y, batch_size, shuffle=True):
    # Yields mini-batches from tensors that already live on the GPU.
    idx = torch.randperm(len(X), device=X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        j = idx[i:i + batch_size]
        yield X[j], Y[j]

# A few training images (un-normalised for display)
fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for ax, i in zip(axes, range(10)):
    img = (X_train[i].cpu() * std[0] + mean[0]).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(img); ax.set_title(classes[Y_train[i]], fontsize=9); ax.axis('off')
plt.show()

---
## Task 1: Backpropagation

### 1(a) Layers with explicit forward/backward, and an MLP

Each layer stores what it needs during `forward` and uses it in `backward`. `backward` receives $\frac{\partial L}{\partial \text{output}}$ (the *upstream gradient*) and returns $\frac{\partial L}{\partial \text{input}}$, so the gradient can be passed from layer to layer by the chain rule.

**Linear**: $Y = XW + b$, with $X \in \mathbb{R}^{N\times d_\text{in}}$, $W \in \mathbb{R}^{d_\text{in}\times d_\text{out}}$, $b \in \mathbb{R}^{d_\text{out}}$. Given $G = \frac{\partial L}{\partial Y}$:

$$\frac{\partial L}{\partial W} = X^\top G, \qquad \frac{\partial L}{\partial b} = \sum_{n=1}^{N} G_{n,:}, \qquad \frac{\partial L}{\partial X} = G\,W^\top$$

A quick way to remember these is to check the shapes: $\frac{\partial L}{\partial W}$ must have the same shape as $W$, which is $(d_\text{in}\times d_\text{out})$, and the only product of $X$ and $G$ with that shape is $X^\top G$. The bias is added to every row of the batch, so its gradient is the sum over the batch.

**ReLU**: $y = \max(0, x)$, so $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\cdot \mathbb{1}[x > 0]$. The gradient passes through where the input was positive and is blocked everywhere else.

**Softmax cross-entropy**: with logits $z$, $p = \text{softmax}(z)$ and the mean loss $L = -\frac{1}{N}\sum_n \log p_{n, y_n}$:

$$\frac{\partial L}{\partial z} = \frac{1}{N}\left(p - \text{onehot}(y)\right)$$

Softmax and cross-entropy are combined into one layer because the joint gradient is this simple expression, and because computing $\log$-softmax as $z - \max z - \log\sum e^{z - \max z}$ avoids overflow in `exp`.

**Initialisation**: weights are drawn from $\mathcal{N}(0, 2/d_\text{in})$ (He/Kaiming initialisation). A ReLU zeros about half its inputs, and the factor 2 compensates for that, so the activation scale stays roughly constant from layer to layer.

In [ ]:
class Linear:
    # y = x @ W + b    x: (N, d_in)   W: (d_in, d_out)   b: (d_out,)
    def __init__(self, d_in, d_out, std=None, dtype=torch.float32):
        std = math.sqrt(2.0 / d_in) if std is None else std   # He init by default
        self.W = torch.randn(d_in, d_out, device=device, dtype=dtype) * std
        self.b = torch.zeros(d_out, device=device, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x                       # cache the input for backward
        return x @ self.W + self.b

    def backward(self, dout):            # dout = dL/dy, shape (N, d_out)
        self.dW = self.x.T @ dout        # (d_in, N) @ (N, d_out) -> (d_in, d_out)
        self.db = dout.sum(dim=0)        # b is added to every row -> sum over the batch
        return dout @ self.W.T           # dL/dx, shape (N, d_in)

    def params(self): return [self.W, self.b]
    def grads(self):  return [self.dW, self.db]


class ReLU:
    def forward(self, x):
        self.mask = x > 0                # remember where the input was positive
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask          # gradient flows only where x > 0

    def params(self): return []
    def grads(self):  return []


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        shifted = logits - logits.max(dim=1, keepdim=True).values      # avoids exp overflow
        log_probs = shifted - shifted.exp().sum(dim=1, keepdim=True).log()
        self.probs = log_probs.exp()
        self.y = y
        return -log_probs[torch.arange(len(y)), y].mean()

    def backward(self):
        N = len(self.y)
        dlogits = self.probs.clone()
        dlogits[torch.arange(N), self.y] -= 1                          # p - onehot(y)
        return dlogits / N                                             # /N because the loss is a mean


class MLP:
    # sizes = [3072, 256, 10] gives Linear(3072,256) -> act -> Linear(256,10)
    def __init__(self, sizes, activation=ReLU, std=None, dtype=torch.float32):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], std=std, dtype=dtype))
            if i < len(sizes) - 2:                                     # no activation after the last layer
                self.layers.append(activation())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        dout = self.loss_fn.backward()
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

    def params(self): return [p for layer in self.layers for p in layer.params()]
    def grads(self):  return [g for layer in self.layers for g in layer.grads()]

In [ ]:
# Sanity check: one forward and backward pass on a small batch
set_seed()
mlp = MLP([3 * 32 * 32, 256, 128, 10])
xb, yb = X_train[:64].flatten(1), Y_train[:64]

loss = mlp.loss(xb, yb)
mlp.backward()
print(f'initial loss = {loss.item():.4f}   (ln 10 = {math.log(10):.4f})')
for i, (p, g) in enumerate(zip(mlp.params(), mlp.grads())):
    print(f'param {i}: shape {tuple(p.shape)!s:14}  grad shape {tuple(g.shape)!s:14}  grad norm {g.norm():.4f}')

**Interpretation.** Every gradient has the same shape as its parameter, which is necessary for a gradient-descent update $\theta \leftarrow \theta - \eta\,\nabla_\theta L$. A network that knows nothing about the classes and gives each of the 10 classes probability $1/10$ would have loss $\ln 10 \approx 2.30$, so this is the reference value for an untrained model. Our initial loss is somewhat higher. With He initialisation the initial logits are not all zero; their standard deviation is of order 1, so the softmax is not perfectly uniform and the network is slightly confident at random. A loss far above $\ln 10$ (e.g. 10 or more) would mean the initial weights are much too large. The gradients themselves are verified in Task 1(b).